# Ablation: Retrieval P/R/F1

Evaluates candidate retrieval on the 579-pair corpus (131 YES from `real_contradiction` + 448 NO from `contradiction_binary`) loaded by `load_triples_to_neo4j.ipynb`.

- **RQ1 (Structural Filter)**: S-SR vs S-SO vs S-Union
- **RQ2 (Retrieval Method)**: R-Struct vs R-Vector@K vs R-Union@K. Vector uses **per-claim top-K** with each claim acting as a simulated query (models the production setup where a user-submitted query is vectorized and top-K neighbors are returned).
- **Per-type recall**: breakdown by `contradiction_type` for the 131 YES pairs (WK subtypes collapsed into `WK`).

**Evaluation protocol** - pool = all 579 `pair_id`s. A method is evaluated by the set of `pair_id`s it retrieves as candidate contradictions.

- `TP` = retrieved ∩ YES
- `FP` = retrieved ∩ NO
- `FN` = YES \ retrieved

Prerequisites: `load_triples_to_neo4j.ipynb` must have been run so the graph has `:Claim` nodes (with embedding) and `[:RELATION]` edges carrying `pair_id` and `contradiction_label`.

In [22]:
from collections import defaultdict

import numpy as np
from langchain_neo4j import Neo4jGraph
from pydantic import SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

In [23]:
class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",
    )
    neo4j_uri: str = "bolt://localhost:7687"
    neo4j_user: str = "neo4j"
    neo4j_password: SecretStr


settings = Settings()
graph = Neo4jGraph(
    url=settings.neo4j_uri,
    username=settings.neo4j_user,
    password=settings.neo4j_password.get_secret_value(),
)
print("Connected")

Connected


## Load gold labels, claims, and embedding matrix

Fetch every `:Claim` from the graph in one query. Build:

- `claims_by_pair[pair_id]` - dict of side ∈ {`t`, `h`} → claim record
- `gold_yes`, `gold_no` - sets of pair_ids
- `embeddings` - L2-normalized N×384 matrix for cosine-via-dot-product
- `cid_to_idx` - maps claim_id to its row in the embedding matrix

In [24]:
claim_rows = graph.query(
    """
    MATCH (c:Claim)
    RETURN c.claim_id AS claim_id, c.pair_id AS pair_id, c.side AS side,
           c.contradiction_label AS label, c.contradiction_type AS type,
           c.embedding AS embedding
    """
)


def normalize_type(t: str | None) -> str | None:
    if t and t.startswith("WK"):
        return "WK"
    return t


claim_ids = [r["claim_id"] for r in claim_rows]
cid_to_idx = {cid: i for i, cid in enumerate(claim_ids)}
embeddings = np.array([r["embedding"] for r in claim_rows], dtype=np.float32)
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)

claims_by_pair: dict[str, dict[str, dict]] = defaultdict(dict)
for r in claim_rows:
    claims_by_pair[r["pair_id"]][r["side"]] = r

all_pairs = set(claims_by_pair.keys())
gold_yes = {pid for pid, sides in claims_by_pair.items() if any(s["label"] == "YES" for s in sides.values())}
gold_no = all_pairs - gold_yes
pair_type = {pid: normalize_type(next(iter(sides.values()))["type"]) for pid, sides in claims_by_pair.items()}

print(f"Claims: {len(claim_rows)}  |  Pairs: {len(all_pairs)}")
print(f"Gold YES: {len(gold_yes)}  |  Gold NO: {len(gold_no)}")
print(f"Embedding matrix: {embeddings.shape}")

Claims: 1158  |  Pairs: 579
Gold YES: 131  |  Gold NO: 448
Embedding matrix: (1158, 384)


## Metrics helpers

In [25]:
def compute_metrics(candidates: set[str]) -> dict:
    tp = len(candidates & gold_yes)
    fp = len(candidates & gold_no)
    fn = len(gold_yes - candidates)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}


def print_metrics(rows: list[tuple[str, dict]]) -> None:
    print(f"{'Method':16} {'TP':>4} {'FP':>4} {'FN':>4} {'P':>7} {'R':>7} {'F1':>7}")
    print("-" * 52)
    for name, m in rows:
        print(f"{name:16} {m['tp']:>4} {m['fp']:>4} {m['fn']:>4} {m['precision']:>7.3f} {m['recall']:>7.3f} {m['f1']:>7.3f}")

## RQ1: Structural filter (S-SR / S-SO / S-Union)

- **S-SR**: same subject + predicate, different object - fires on functional-predicate disagreements (numeric, temporal, WK-location).
- **S-SO**: same subject + object, different predicate - fires on relation-switching (antonym, negation).
- **S-Union**: either condition.

In [26]:
def s_sr_candidates() -> set[str]:
    rows = graph.query(
        """
        MATCH (s:Entity)-[r1:RELATION]->(o1:Entity),
              (s)-[r2:RELATION]->(o2:Entity)
        WHERE r1.pair_id = r2.pair_id
          AND r1.side = 't' AND r2.side = 'h'
          AND r1.predicate = r2.predicate
          AND o1.name <> o2.name
        RETURN DISTINCT r1.pair_id AS pair_id
        """
    )
    return {r["pair_id"] for r in rows}


def s_so_candidates() -> set[str]:
    rows = graph.query(
        """
        MATCH (s:Entity)-[r1:RELATION]->(o:Entity),
              (s)-[r2:RELATION]->(o)
        WHERE r1.pair_id = r2.pair_id
          AND r1.side = 't' AND r2.side = 'h'
          AND r1.predicate <> r2.predicate
        RETURN DISTINCT r1.pair_id AS pair_id
        """
    )
    return {r["pair_id"] for r in rows}


s_sr = s_sr_candidates()
s_so = s_so_candidates()
s_union = s_sr | s_so

print_metrics(
    [
        ("S-SR", compute_metrics(s_sr)),
        ("S-SO", compute_metrics(s_so)),
        ("S-Union", compute_metrics(s_union)),
    ]
)

Method             TP   FP   FN       P       R      F1
----------------------------------------------------
S-SR               17    5  114   0.773   0.130   0.222
S-SO                1   16  130   0.059   0.008   0.014
S-Union            17   21  114   0.447   0.130   0.201


## RQ2: Retrieval method (R-Struct / R-Vector@K / R-Union@K)

**Query-simulated top-K retrieval** (models deployment: the production system vectorizes a user query `Q` and returns its top-K nearest `:Claim` nodes).

Offline, we don't have real user queries, so **each claim in the corpus plays the role of a hypothetical query in turn**. For each labeled pair `P = (t_P, h_P)`:

1. Treat `t_P`'s embedding as the query; retrieve top-K nearest claims (self-match masked).
2. If `h_P` is in that top-K, `P` is retrieved.
3. Symmetric check: also try `h_P` as query; if `t_P` is in its top-K, `P` is retrieved.

This matches how the system works at inference time - the only difference is that offline the "query" happens to be an existing claim's embedding, whereas at inference the query is the user's natural-language input encoded by the same SBERT model.

- `R-Struct` = `S-Union` from RQ1.
- `R-Vector@K` = pairs where the partner appears in the K-neighborhood (either direction).
- `R-Union@K` = `R-Struct ∪ R-Vector@K`.

K is swept on a per-claim scale: `{1, 3, 5, 10, 20, 50}`.

> **Paper note**: §3.3 Step 4b currently reads "top-k pairs". Clearer wording to match this implementation: "for each claim, retrieve top-K cosine neighbors; a candidate pair is a claim + one of its K neighbors."

In [27]:
sim = embeddings @ embeddings.T


def vector_candidates(k: int) -> set[str]:
    """Per-claim top-K retrieval (simulated query). A labeled pair (t_P, h_P) is retrieved at K
    iff h_P is in t_P's top-K cosine neighbors, OR t_P is in h_P's top-K (symmetric check).
    Self-match is masked via -inf."""
    retrieved = set()
    for pair_id, sides in claims_by_pair.items():
        if "t" not in sides or "h" not in sides:
            continue
        t_idx = cid_to_idx[sides["t"]["claim_id"]]
        h_idx = cid_to_idx[sides["h"]["claim_id"]]
        t_scores = sim[t_idx].copy()
        t_scores[t_idx] = -np.inf
        if h_idx in np.argpartition(-t_scores, k - 1)[:k]:
            retrieved.add(pair_id)
            continue
        h_scores = sim[h_idx].copy()
        h_scores[h_idx] = -np.inf
        if t_idx in np.argpartition(-h_scores, k - 1)[:k]:
            retrieved.add(pair_id)
    return retrieved


rows = [("R-Struct", compute_metrics(s_union))]
for k in [1, 3, 5, 10, 20, 50]:
    r_vec = vector_candidates(k)
    r_union = s_union | r_vec
    rows.append((f"R-Vector@{k}", compute_metrics(r_vec)))
    rows.append((f"R-Union@{k}", compute_metrics(r_union)))

print_metrics(rows)

Method             TP   FP   FN       P       R      F1
----------------------------------------------------
R-Struct           17   21  114   0.447   0.130   0.201
R-Vector@1         72  242   59   0.229   0.550   0.324
R-Union@1          76  243   55   0.238   0.580   0.338
R-Vector@3         93  349   38   0.210   0.710   0.325
R-Union@3          94  350   37   0.212   0.718   0.327
R-Vector@5        109  395   22   0.216   0.832   0.343
R-Union@5         109  396   22   0.216   0.832   0.343
R-Vector@10       123  420    8   0.227   0.939   0.365
R-Union@10        123  420    8   0.227   0.939   0.365
R-Vector@20       130  431    1   0.232   0.992   0.376
R-Union@20        130  431    1   0.232   0.992   0.376
R-Vector@50       131  438    0   0.230   1.000   0.374
R-Union@50        131  438    0   0.230   1.000   0.374


## Recall by contradiction type (YES pairs only)

Only YES pairs carry a meaningful `type`. Shows where each method fires well and where it misses. NO pairs have `type = "none"` and are excluded here.

In [28]:
def recall_by_type(candidates: set[str]) -> dict[str, tuple[int, int]]:
    bucket: dict[str, list[int]] = defaultdict(lambda: [0, 0])
    for pid in gold_yes:
        t = pair_type[pid]
        bucket[t][1] += 1
        if pid in candidates:
            bucket[t][0] += 1
    return {t: (tp, total) for t, (tp, total) in bucket.items()}


TYPE_K = 10  # pick K from the RQ2 sweep; adjust if you want a different operating point

methods_for_type = [
    ("S-SR", s_sr),
    ("S-SO", s_so),
    ("S-Union", s_union),
    (f"R-Vector@{TYPE_K}", vector_candidates(TYPE_K)),
    (f"R-Union@{TYPE_K}", s_union | vector_candidates(TYPE_K)),
]

types = sorted({pair_type[p] for p in gold_yes})

header = f"{'Type':16} {'N':>4}  " + "  ".join(f"{name:>14}" for name, _ in methods_for_type)
print(header)
print("-" * len(header))
for t in types:
    total = sum(1 for p in gold_yes if pair_type[p] == t)
    cells = []
    for _, cands in methods_for_type:
        tp, _ = recall_by_type(cands).get(t, (0, 0))
        cells.append(f"{tp:>2}/{total:<2} {tp / max(total, 1) * 100:>4.0f}%")
    print(f"{t:16} {total:>4}  " + "  ".join(f"{c:>14}" for c in cells))

Type                N            S-SR            S-SO         S-Union     R-Vector@10      R-Union@10
-----------------------------------------------------------------------------------------------------
WK                 17      4/17   24%      0/17    0%      4/17   24%     17/17  100%     17/17  100%
antonym            12      2/12   17%      0/12    0%      2/12   17%     10/12   83%     10/12   83%
factive             9      0/9     0%      0/9     0%      0/9     0%      8/9    89%      8/9    89%
lexical            28      2/28    7%      0/28    0%      2/28    7%     27/28   96%     27/28   96%
negation           23      1/23    4%      0/23    0%      1/23    4%     21/23   91%     21/23   91%
numeric            38      7/38   18%      1/38    3%      7/38   18%     36/38   95%     36/38   95%
structure           4      1/4    25%      0/4     0%      1/4    25%      4/4   100%      4/4   100%
